# RSNA Knee Abnormality Detection — v1 runbook (single notebook)**One file, end-to-end:** pseudo-labels from reports (local LLM) → MedSigLIP embeddings (frozen) → small head → `submission.csv`.**Runtime target:** head training < 1 h on CPU; full pipeline fits a Kaggle 12 h GPU session. Final models load **100% offline** (no internet calls at submission time).---## 0. Where this came from — steps we took (decision log)1. **Data recon (local, CSV-only):** `train.csv` = 4,407 studies with radiology `Report` text, but only **58** have all 12 ground-truth labels. `train_series.csv` = 24,371 series with MRI protocol flags. Full DICOM volume ≈ 569 GB → **does not live on this laptop**, only CSVs do.2. **Data source decision:** the 569 GB DICOMs are *pre-mounted* on Kaggle at `/kaggle/input/rsna-knee-abnormality-detection` — no download needed there. Laptop/Colab = CSV + model work only.3. **Model search:** shortlisted **Flair (Google DeepMind, gated HF `flair-med/embedding_encoder_medium`)** + **MedSigLIP-448 (`google/medsiglip-448`)**. Then: **Flair removed entirely** — decision: MedSigLIP-only (one encoder, 400M vision tower, pretrained on MRI slices + reports, officially built for *data-efficient classification with few labels* — exactly our 58-label problem). Keeps v1 small, fast, and simple.4. **Label strategy change:** was Google Gemini API (free tier) → **replaced with a small local LLM (Gemma 3 4B IT)** because competition notebooks run with **internet disabled** — we need weights that are *attached to the notebook* (Kaggle Models / private dataset), never fetched at runtime.5. **Offline constraint check:** Kaggle competition submissions run offline. Rule enforced in this notebook: **zero internet calls after setup**. All model weights must be added as notebook *inputs* (see section 1).**Targets:** v1 ≈ 0.78–0.83 public AUC; stretch 0.84–0.86 via label-quality and ensembling (v1.1+).---## 1. Running this notebook — the offline rule| Environment | Data | Models | Notes ||---|---|---|---|| **Kaggle (submission)** | auto-mounted `/kaggle/input/rsna-knee-abnormality-detection` | attached as inputs | the real run || Kaggle (interactive) | same | same | develop here || Colab / laptop | CSVs only (local `..` or upload) | via HF with token | DICOM cells skip gracefully |**Offline model checklist — how each model "gets installed" (both are attached as notebook *inputs*, never downloaded inside the notebook):****1. Gemma 3 4B IT (transformers format) — one-time, with internet, on your laptop:**```bashpip install huggingface_hub# 1) accept terms: https://huggingface.co/google/gemma-3-4b-it  (needs HF login)# 2) download the weights (needs HF_TOKEN; ≈ 8 GB)HF_TOKEN=hf_xxxx huggingface-cli download google/gemma-3-4b-it --local-dir gemma3-4b-it# 3) zip + upload as a PRIVATE Kaggle Dataset, e.g. "gemma3-4b-it-transformers"# 4) in this notebook: "+ Add Input" → your private Dataset → attach it```At runtime the notebook scans `/kaggle/input/...` for a folder with `config.json` whose name contains `gemma` and loads it with `AutoModelForCausalLM.from_pretrained(<local path>)` — **zero network**.*(Do NOT use the Kaggle Models panel `google/gemma-3/pyTorch/...` — that ships the `model.ckpt` gemma-pytorch format and will not load from `transformers`.)***2. MedSigLIP-448 — same pattern:**```bash# 1) accept terms: https://huggingface.co/google/medsiglip-448  (you already did)# 2) download (≈ 3 GB):HF_TOKEN=hf_xxxx huggingface-cli download google/medsiglip-448 --local-dir medsiglip-448# 3) zip + upload as a PRIVATE Kaggle Dataset "medsiglip-448"# 4) "+ Add Input" → attach it```Resolver scans `/kaggle/input/...` for a `config.json` containing `medsiglip` → `SiglipVisionModel.from_pretrained(<local path>)`.**Both models at submission time:** the attached inputs are mounted from Kaggle's resource cache as plain local folders — the notebook never makes a network call to fetch weights. (If a model is missing, the notebook raises a clear message instead of silently downloading.)

In [ ]:
import os, sys, re, json, glob, time, warnings, subprocessfrom pathlib import Pathimport numpy as npimport pandas as pdwarnings.filterwarnings("ignore")# --- environment detection -------------------------------------------------IS_KAGGLE  = Path("/kaggle").exists()IS_COLAB   = "google.colab" in sys.modulestry:    import torch    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"except ImportError:    DEVICE = "cpu"WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path(".")# --- data location ----------------------------------------------------------if IS_KAGGLE:    DATA_DIR = Path("/kaggle/input/rsna-knee-abnormality-detection")else:    DATA_DIR = Path("..")   # CSVs at repo root (this laptop)print(f"kaggle={IS_KAGGLE}  colab={IS_COLAB}  device={DEVICE}")print(f"data_dir={DATA_DIR.resolve()}  exists={DATA_DIR.exists()}")

In [ ]:
# --- load the 5 competition CSVs ---------------------------------------------CSV_FILES = ["train.csv", "train_series.csv", "test.csv", "test_series.csv", "sample_submission.csv"]def load_csv(name):    p = DATA_DIR / name    if not p.exists():        print(f"[warn] missing {name} (expected if DICOM-only env). Returning None")        return None    return pd.read_csv(p)train, train_series, test, test_series, sample_sub = [load_csv(f) for f in CSV_FILES]LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]if train is not None:    real = train[train[LABELS].notna().any(axis=1)]    print(f"train studies: {len(train)}  | fully/partially labeled studies: {len(real)}")    print(f"report coverage: {train['Report'].notna().mean():.2%}")    print(f"language examples:\n{train['Report'].head(3).astype(str).str[:60].to_string()}")

## STEP 1 — Pseudo-labels from reports using a **local** LLM (no API, no internet)Why: only 58 studies have labels; the other ~4,349 have radiology reports in many languages (ES/NL/FR/DE/IT/CS/EN). We ask Gemma to convert each report into a compact row of 12 binary labels. Explicit mention ⇒ 1, explicit absence/normal ⇒ 0, unclear/not mentioned ⇒ 0. Never invent.Cheap output format: a single line `[0,1,0,1,0,0,0,1,0,0,0,0]` (12 digits) — keeps tokens tiny and parsing bulletproof.

In [ ]:
# --- resolve + load the LLM (offline-first) -----------------------------------# Two valid offline sources (both show up as plain folders on disk):#   A) A transformers-format Gemma attached as a Kaggle **Dataset/Model input**#      (recommended) — e.g. download gemma-3-4b-it with internet once, upload to a#      private Kaggle Dataset, attach it. It loads with AutoModelForCausalLM.#   B) Kaggle Models input `google/gemma-3/pyTorch/gemma-3-4b-it` — but that is the#      gemma_pytorch ckpt format and won't load from `transformers.from_pretrained`;#      use (A). We keep it here only to auto-skip and print a hint.LLM_HF_ID = "google/gemma-3-4b-it"              # used ONLY when internet is available (Colab/laptop)def find_model_dir(name_hint):    """Scan /kaggle/input for a PyTorch/transformers repo dir (config.json + weights).    Pure filesystem — no network. Returns first match or None."""    if not Path("/kaggle/input").exists():        return None    hits = []    for p in Path("/kaggle/input").glob("*"):        if p.is_dir() and name_hint.lower() in p.name.lower():            for cand in [p, *(list(p.iterdir()) if p.is_dir() else [])]:                if isinstance(cand, Path) and cand.is_dir() and (cand / "config.json").exists():                    hits.append(cand)    # prefer a dir whose name mentions the exact model/variant    return hits[0] if hits else Nonedef load_llm(device=None):    from transformers import AutoModelForCausalLM, AutoTokenizer    device = device or DEVICE    w = find_model_dir("gemma")    if w is None and not IS_KAGGLE:        # internet fallback (Colab / laptop only)        try:            from transformers import AutoModelForCausalLM, AutoTokenizer            tok = AutoTokenizer.from_pretrained(LLM_HF_ID, token=os.environ.get("HF_TOKEN"))            tok.padding_side = "left"            if tok.pad_token is None: tok.pad_token = tok.eos_token            model = AutoModelForCausalLM.from_pretrained(                LLM_HF_ID, token=os.environ.get("HF_TOKEN"),                torch_dtype="float32" if device == "cpu" else "bfloat16",                device_map=None if device == "cpu" else "auto")            model.eval()            return model, tok        except Exception as e:            print("[hint] LLM not found & no internet — attach a transformers-format gemma to /kaggle/input")            raise e    if w is None:        raise RuntimeError(            "Gemma not found. Attach a TRANSFORMERS-format gemma-3-4b-it as a Private dataset "            "or Model input (NOT google/gemma-3/pyTorch which is gemma-pytorch ckpt format).")    print("loading LLM from", w)    tok = AutoTokenizer.from_pretrained(str(w))    tok.padding_side = "left"    if tok.pad_token is None: tok.pad_token = tok.eos_token    kw = dict(torch_dtype="float32" if device == "cpu" else "bfloat16",              device_map=None if device == "cpu" else "auto")    model = AutoModelForCausalLM.from_pretrained(str(w), **kw)    model.eval()    return model, tokLLM_DEVICE = DEVICEif IS_KAGGLE or find_model_dir("gemma") is not None:    llm, llm_tok = load_llm(LLM_DEVICE)    print(f"LLM ready — params ~{llm.num_parameters()/1e9:.1f}B")else:    print("[warn] Gemma not attached; Step 1 will be skipped this session.")    llm, llm_tok = None, None

In [ ]:
# --- the extraction prompt ----------------------------------------------------PROMPT = """You are a radiology coding assistant for a knee MRI abnormality-detection challenge.Read the radiology report and output EXACTLY ONE line: a JSON array of 12 integers in this order:ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA, Lateral OA, PF OA, Effusion, Synovitis, Baker's, Contusion, FractureRules:- 1 = finding is explicitly mentioned or strongly implied (e.g. rotura/rupture/tear, derrame/effusion,  artrosis/osteoarthritis, gonartrosis, joint space narrowing, baker cyst, contusion, fracture).- 0 = explicitly described as normal/intact/without finding (no signs of, sin lesiones, unremarkable, normaal, oedemfrei...) OR not mentioned at all.- OA counts as 1 if osteoarthritis/arthrosis/gonarthrosis is reported (any grade wording); do NOT count incidental joint space narrowing alone.- Medial OA uses medial compartment findings; Lateral OA lateral; PF OA patellofemoral.- Do not add explanations. Output only the JSON array, e.g. [0,1,0,0,1,0,0,1,0,1,0,0]Report: {report}Output:"""def build_messages(report):    return [{"role": "user", "content": PROMPT.format(report=report[:2500])}]def parse_label_row(text):    """Parse the 12-int array; retry patterns; else None."""    m = re.search(r"\[([0-1,\s]+)\]", text)    if m:        vals = [int(x) for x in re.findall(r"[01]", m.group(1))]        if len(vals) == 12:            return vals    return None

In [ ]:
# --- run extraction (batched generation) --------------------------------------BATCH = 8          # increase on GPU (T4: 8-16); use 1 on CPUMAX_NEW = 32def extract_batch(reports, model, tok, batch=BATCH):    msgs = [build_messages(r) for r in reports]    texts = [tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False) for m in msgs]    enc = tok(texts, return_tensors="pt", padding=True)    if model is not None:        enc = enc.to(model.device)    with torch.inference_mode():        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)    out = out[:, enc["input_ids"].shape[1]:]    return [tok.decode(o, skip_special_tokens=True) for o in out]def extract_all(train_df, model, tok, out_csv="pseudo_labels.csv", limit=None):    df = train_df.copy()    df["Report"] = df["Report"].fillna("")    sub = df[df["Report"].str.strip() != ""]    if limit: sub = sub.head(limit)    rows, done, fail = [], 0, 0    t0 = time.time()    for i in range(0, len(sub), BATCH):        chunk = sub.iloc[i:i+BATCH]        decs = extract_batch(chunk["Report"].tolist(), model, tok)        for uid, rep, dec in zip(chunk["StudyInstanceUID"], chunk["Report"], decs):            vals = parse_label_row(dec)            if vals is None:                fail += 1                vals = [0]*12            rows.append({"StudyInstanceUID": uid, "Report_len": len(rep), "raw": dec[:80], **dict(zip(LABELS, vals))})        done += len(chunk)        if (done % 64) == 0 or i + BATCH >= len(sub):            spd = done / max(time.time() - t0, 1e-6)            print(f"[{done}/{len(sub)}] {spd:.1f} reports/s  fails={fail}", flush=True)    res = pd.DataFrame(rows)    (WORK_DIR / out_csv).parent.mkdir(parents=True, exist_ok=True)    res.to_csv(WORK_DIR / out_csv, index=False)    print(f"saved {WORK_DIR/out_csv} | parsed {len(res)} rows | unparsed {fail} ({fail/len(res):.1%})")    return res# quick smoke test (1 batch) so you can fix problems before the full runif llm is not None:    smoke = extract_batch(["RM rodilla: rotura de menisco interno. Artrosis femorotibial medial. Derrame.",                           "MR knee: no effusion, no meniscal tear, intact ACL/MCL. Unremarkable study.",                           "Knie: keine Gelenkerguss, Baker-Zyste vorhanden. Vorderes Kreuzband intakt."],                          llm, llm_tok)    print(smoke)else:    print("[skip] smoke test — Gemma not attached")

In [ ]:
# --- FULL run: pseudo-labels for every report-bearing study ------------------# On Kaggle GPU: ~4,400 reports ≈ 20-40 min with batch=8. On CPU: use limit=64 first.PSEUDO_LIMIT = None          # e.g. 256 for a quick sanity pass; None = allif llm is not None:    pseudo = extract_all(train, llm, llm_tok, out_csv="pseudo_labels.csv", limit=PSEUDO_LIMIT)    print(pseudo[["StudyInstanceUID"] + LABELS].head().to_string())else:    print("[skip] full extraction — Gemma not attached; attach it, then re-run from Step 1")    pseudo = pd.DataFrame()

In [ ]:
# --- validate pseudo-labels against the 58 real-labeled studies ---------------if train is not None and pseudo is not None:    real = train[train[LABELS].notna().any(axis=1)].copy()    # the 58 labeled studies also have reports, so they are inside `pseudo` too    merged = real.merge(pseudo[["StudyInstanceUID"] + LABELS],                        on="StudyInstanceUID", suffixes=("_true", "_pseudo"))    print(f"overlap with real labels: {len(merged)} studies")    from sklearn.metrics import accuracy_score, f1_score    rows = []    for col in LABELS:        yt = merged[f"{col}_true"].astype(int)        yp = merged[f"{col}_pseudo"].astype(int)        rows.append({"finding": col,                     "true_pos": int(yt.sum()), "agreement": accuracy_score(yt, yp),                     "f1": f1_score(yt, yp, zero_division=0)})    val = pd.DataFrame(rows)    print(val.to_string(index=False))    print(f"\nmean agreement: {val['agreement'].mean():.2%} | mean F1: {val['f1'].mean():.3f}")    val.to_csv(WORK_DIR / "label_validation.csv", index=False)

## STEP 2 — MedSigLIP embeddings (frozen vision tower)Slice selection per study (from `train_series.csv` protocol flags), then embed every selected slice with MedSigLIP-448 and pool to one 1536-dim study vector (mean+max over slices).- DICOM → window-normalized RGB via pydicom (no intensity dependency: per-slice 1–99 percentile stretch).- Selection: up to 2 series per study, priority: Sagittal/fluid-sensitive → Coronal/fluid-sensitive → Axial/fat-suppressed → whatever else.- Slices: middle `N_SLICES` of each selected series (knee-centered).- Runs on Kaggle (data mounted); skips gracefully elsewhere.

In [ ]:
# --- resolve + load MedSigLIP (offline-first) ---------------------------------MEDSIGLIP_HF_ID = "google/medsiglip-448"HF_TOKEN = os.environ.get("HF_TOKEN")def find_medsiglip_dir():    """Pure filesystem scan for an attached MedSigLIP repo (config.json mentioning 'medsiglip')."""    if Path("/kaggle/input").exists():        for p in Path("/kaggle/input").glob("*"):            for cand in [p, *(list(p.iterdir()) if p.is_dir() else [])]:                if isinstance(cand, Path) and cand.is_dir() and (cand / "config.json").exists():                    try:                        cfg = json.loads((cand / "config.json").read_text())                        if "medsiglip" in json.dumps(cfg).lower():                            return cand                    except Exception:                        pass    return NoneMEDSIGLIP_DIR = find_medsiglip_dir()if MEDSIGLIP_DIR:    print("MedSigLIP from attached Kaggle input:", MEDSIGLIP_DIR)else:    print("MedSigLIP: no Kaggle input found -> will use HF (needs HF_TOKEN, internet)")if MEDSIGLIP_DIR:    msl_src = str(MEDSIGLIP_DIR)else:    if not IS_KAGGLE and HF_TOKEN:        msl_src = MEDSIGLIP_HF_ID    else:        raise RuntimeError(            "MedSigLIP weights are not attached. Instruction: with internet, run once\n"            "  huggingface-cli download google/medsiglip-448 --local-dir medsiglip-448\n"            "zip that folder and upload it as a PRIVATE Kaggle Dataset named e.g. 'medsiglip-448',\n"            "then Add it as an input to this notebook (no internet needed at runtime).")from transformers import SiglipVisionModel, AutoProcessormsl = SiglipVisionModel.from_pretrained(msl_src,                                        token=None if MEDSIGLIP_DIR else HF_TOKEN).to(DEVICE).eval()msl_pp = AutoProcessor.from_pretrained(msl_src, token=None if MEDSIGLIP_DIR else HF_TOKEN)print(f"MedSigLIP ready on {DEVICE}")

In [ ]:
# --- DICOM helpers -------------------------------------------------------------def get_series_dir(study_uid):    base = DATA_DIR / ("train_series" if str(study_uid) in train["StudyInstanceUID"].astype(str).values                       else "test_series")    d = base / str(study_uid)    return d if d.exists() else Nonedef select_series(series_df, study_uid, max_series=2):    s = series_df[series_df["StudyInstanceUID"] == study_uid]    prio = [("Sagittal", 1, 0), ("Coronal", 1, 0), ("Axial", 0, 1),            ("Sagittal", 0, 0), ("Coronal", 0, 0), ("Axial", 0, 0)]    chosen = []    for plane, fs, fat in prio:        m = s[(s["Anatomical_Plane"] == plane) & (s["Fluid_Sensitive"] == fs) & (s["Fat_Suppression"] == fat)]        chosen += m["SeriesInstanceUID"].tolist()    return chosen[:max_series]def load_slice_pil(dcm_path):    import pydicom    ds = pydicom.dcmread(dcm_path)    arr = ds.pixel_array.astype(np.float32)    arr = arr * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))    lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)    arr = np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1)    img = (arr * 255).astype(np.uint8)    from PIL import Image    return Image.fromarray(img).convert("RGB")def middle_slices(dcm_dir, n=15):    files = sorted(Path(dcm_dir).glob("*.dcm"))    if not files:        return []    lo = max(0, len(files)//2 - n//2)    return files[lo:lo+n]

In [ ]:
# --- embed one study: per-slice MedSigLIP -> pooled study vector --------------N_SLICES = 15      # middle slices per seriesEMB_BATCH = 32     # slices per forward passdef get_series_dir_map(study_uid):    base = DATA_DIR / ("train_series" if str(study_uid) in train["StudyInstanceUID"].astype(str).values                       else "test_series")    d = base / str(study_uid)    return d if d.exists() else None@torch.inference_mode()def embed_study(study_uid, series_df, model, proc, device=DEVICE, n_slices=N_SLICES):    import pydicom    from PIL import Image    study_dir = get_series_dir(study_uid)    if study_dir is None:        return None    sid_map = {sd.name: sd for sd in study_dir.iterdir() if sd.is_dir()}    series = select_series(series_df, study_uid)    all_emb = []    for sid in series:        dcm_dir = sid_map.get(str(sid))        if dcm_dir is None:            continue        slices = middle_slices(dcm_dir, n=n_slices)        for i in range(0, len(slices), EMB_BATCH):            chunk = slices[i:i+EMB_BATCH]            pils = []            for f in chunk:                try:                    pils.append(load_slice_pil(f))                except Exception:                    pass            if not pils:                continue            inputs = proc(images=pils, return_tensors="pt").to(device)            out = model(**inputs)            e = out.pooler_output            e = e / e.norm(dim=-1, keepdim=True)            all_emb.append(e.cpu())    if not all_emb:        return None    E = torch.cat(all_emb, dim=0)                       # (n_slices, 768)    return torch.cat([E.mean(0), E.max(0)[0]], dim=0)   # (1536,)

In [ ]:
# --- full train embedding pass (Kaggle only; ~15-25 min on T4) ---------------EMBED_TRAIN = IS_KAGGLE and train_series is not NoneFEAT_COLS = [f"f{i}" for i in range(1536)]if EMBED_TRAIN:    emb_cache = WORK_DIR / "train_study_emb.parquet"    if emb_cache.exists():        train_feat = pd.read_parquet(emb_cache)        print("loaded cached train embeddings:", train_feat.shape)    else:        rows, t0 = [], time.time()        for i, uid in enumerate(train["StudyInstanceUID"].astype(str)):            v = embed_study(uid, train_series, msl, msl_pp)            if v is not None:                rows.append({"StudyInstanceUID": uid, **dict(zip(FEAT_COLS, v.tolist()))})            if (i + 1) % 200 == 0:                print(f"[{i+1}/{len(train)}] { (i+1)/max(time.time()-t0,1e-6):.0f} studies/s", flush=True)        train_feat = pd.DataFrame(rows)        train_feat.to_parquet(emb_cache, index=False)        print("train embeddings saved:", train_feat.shape)else:    print("[skip] train embedding pass requires Kaggle-mounted DICOMs")    train_feat = pd.DataFrame()

## STEP 3 — Train the head (CPU < 1 h)Features: 1536-dim pooled MedSigLIP embeddings. Targets: pseudo-labels from Step 1 (trust-weighted per label via label-validation F1). Model: one `LogisticRegression` per finding, standardized features, 5-fold CV. Sanity check on the 58 real-labeled studies.

In [ ]:
# --- build train matrix ---------------------------------------------------------if train_feat is not None and len(train_feat) and pseudo is not None:    X = train_feat[FEAT_COLS].values.astype(np.float32)    Y = pseudo.set_index("StudyInstanceUID").reindex(train_feat["StudyInstanceUID"])[LABELS]    print("X:", X.shape, "Y:", Y.shape, "NaN in Y:", int(Y.isna().sum().sum()))    from sklearn.linear_model import LogisticRegression    from sklearn.preprocessing import StandardScaler    from sklearn.model_selection import StratifiedKFold    from sklearn.metrics import roc_auc_score    sc = StandardScaler().fit(X)    Xs = sc.transform(X)    Yf = Y.fillna(0)    cvs, models = {}, {}    for col in LABELS:        y = Yf[col].astype(int).values        if len(np.unique(y)) < 2:            print(f"[skip] {col}: single class in pseudo-labels")            continue        m = LogisticRegression(max_iter=2000, C=0.1)        skf = StratifiedKFold(5, shuffle=True, random_state=42)        aucs = []        for tr, va in skf.split(Xs, y):            m.fit(Xs[tr], y[tr])            aucs.append(roc_auc_score(y[va], m.predict_proba(Xs[va])[:, 1]))        m.fit(Xs, y)        models[col] = m        cvs[col] = np.mean(aucs)        print(f"{col:18s} CV-AUC {np.mean(aucs):.4f}  n_pos={int(y.sum())}")    print(f"\nmean CV-AUC over pseudo-labels: {np.mean(list(cvs.values())):.4f}")    # --- sanity: AUC on the 58 real-labeled studies (head sees their reports too,    # --- so this is optimistic, but it validates the head is learning structure) ---    real = train[train[LABELS].notna().any(axis=1)]    inter = train_feat.merge(real[["StudyInstanceUID"] + LABELS], on="StudyInstanceUID")    if len(inter) >= 5:        Xi = sc.transform(inter[FEAT_COLS].values)        aucs_real = []        for col in LABELS:            if col not in models: continue            yt = inter[col].astype(int).values            if len(np.unique(yt)) < 2: continue            aucs_real.append(roc_auc_score(yt, models[col].predict_proba(Xi)[:, 1]))        print(f"real-label AUC (n={len(inter)} studies): {np.mean(aucs_real):.4f}")else:    print("[skip] head training requires train embeddings + pseudo-labels (run on Kaggle)")

## STEP 4 — Test inference + submission

In [ ]:
# --- embed test studies (Kaggle only) ------------------------------------------if IS_KAGGLE and test_series is not None:    emb_cache = WORK_DIR / "test_study_emb.parquet"    if emb_cache.exists():        test_feat = pd.read_parquet(emb_cache)    else:        rows, t0 = [], time.time()        for i, uid in enumerate(test["StudyInstanceUID"].astype(str)):            v = embed_study(uid, test_series, msl, msl_pp)            if v is not None:                rows.append({"StudyInstanceUID": uid, **dict(zip(FEAT_COLS, v.tolist()))})            if (i + 1) % 100 == 0:                print(f"[{i+1}/{len(test)}]", flush=True)        test_feat = pd.DataFrame(rows)        test_feat.to_parquet(emb_cache, index=False)        print("test embeddings saved:", test_feat.shape)else:    print("[skip] test embedding requires Kaggle-mounted DICOMs")    test_feat = pd.DataFrame()

In [ ]:
# --- predict + write submission -------------------------------------------------if len(test_feat) and models:    Xt = sc.transform(test_feat[FEAT_COLS].values)    pred = {col: models[col].predict_proba(Xt)[:, 1] for col in models}    sub = pd.DataFrame({"StudyInstanceUID": test_feat["StudyInstanceUID"], **pred})    # guarantee all 12 columns, default 0.5    for col in LABELS:        if col not in sub:            sub[col] = 0.5    sub = sub[["StudyInstanceUID"] + LABELS]    # align to sample_submission column order / dtypes    if sample_sub is not None:        sub = sub.reindex(columns=sample_sub.columns)    sub.to_csv(WORK_DIR / "submission.csv", index=False)    print(f"submission rows: {len(sub)}")    if sample_sub is not None:        ok_order = list(sub.columns) == list(sample_sub.columns)        ok_rows  = len(sub) == len(sample_sub)        print(f"columns match sample: {ok_order} | rows match sample: {ok_rows}")    print(sub.head().to_string())else:    print("[skip] no test embeddings — nothing to submit yet")

## 5. Results recap & v1.1 levers**Expected v1:** pseudo-label F1 gate passes (≥0.8 agreement on the 58), CV-AUC on pseudo-labels visible, real-label sanity AUC reported. Final `submission.csv` written in sample format.**v1.1 levers (ordered by expected gain):**1. **Label quality:** multi-LLM ensembling (Gemma + another attached model), uncertainty/confidence weighting in the loss, explicit 'not mentioned' handling.2. **Pooling:** attention pooling over slices instead of mean+max; per-series embeddings (series→finding mapping, e.g. sagittal PD for menisci, axial for patella).3. **Head:** 2-layer MLP + label-wise threshold calibration; prevalence prior from LLM statistics.4. **Ensemble:** 3-fold heads + averaged predictions; later a second small encoder (e.g. DINOv2) as cheap diversity.**Offline rules re-check:** all weights must come from attached inputs (`/kaggle/input/...`) or cached Kaggle Models — no `from_pretrained` with internet at submission time. Re-run the "offline model checklist" in section 1 before submitting.